# 04 — Key Insights

This notebook summarizes the main findings from the exploratory data analysis of the US Top 50 playlist dataset.

The insights focus on:
- Song longevity and chart performance
- Artist dominance
- Popularity trends
- Release format performance
- Relationship between playlist position and popularity

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/us_top50_clean.csv")

df["date"] = pd.to_datetime(df["date"])

print(df.shape)

(27700, 10)


## 1. Song Longevity

Long chart presence indicates that a song maintained visibility on the US Top 50 playlist over an extended period.

In [2]:
# Top 10 songs by chart longevity

song_longevity = (
    df.groupby(["song", "artist"], sort=False)
    .agg(
        chart_days=("date", "nunique"),
        best_position=("position", "min")
    )
    .reset_index()
    .sort_values("chart_days", ascending=False)
)

song_longevity.head(10)

,song,artist,chart_days,best_position
28,Something in the Orange,Zach Bryan,535,9
1,Last Night,Morgan Wallen,491,1
253,I Remember Everything (feat. Kacey Musgraves),Zach Bryan,445,1
5,See You Again (feat. Kali Uchis),"Tyler, The Creator",443,4
313,Stick Season,Noah Kahan,413,1
15,Cruel Summer,Taylor Swift,387,1
499,Lose Control,Teddy Swims,328,5
26,Thinkin’ Bout Me,Morgan Wallen,323,21
527,Beautiful Things,Benson Boone,306,1
16,You Proof,Morgan Wallen,301,9


### Key Finding

The longest-lasting songs maintained a strong presence on the US Top 50 chart over many days. Chart longevity helps identify songs that achieved sustained popularity rather than only short-term success.

In [3]:
# Top 10 artists by playlist appearances

artist_performance = (
    df.groupby("artist")
    .agg(
        playlist_appearances=("song", "count"),
        unique_songs=("song", "nunique"),
        average_position=("position", "mean"),
        average_popularity=("popularity", "mean")
    )
    .reset_index()
    .sort_values("playlist_appearances", ascending=False)
)

artist_performance.head(10)

,artist,playlist_appearances,unique_songs,average_position,average_popularity
242,Taylor Swift,2014,96,24.557100,87.096822
286,Zach Bryan,1854,35,25.619202,86.022114
177,Morgan Wallen,1666,10,30.610444,85.405762
229,Sabrina Carpenter,966,14,17.598344,90.198758
65,Drake,838,35,26.256563,84.608592
40,Chappell Roan,794,6,20.647355,88.664987
266,Travis Scott,760,31,25.744737,88.588158
22,Billie Eilish,729,12,20.362140,92.770919
273,"Tyler, The Creator",705,16,25.259574,88.394326
227,SZA,646,3,26.238390,91.668731


## 2. Artist Performance

Artist performance can be examined using playlist appearances, number of unique songs, average chart position, and average popularity.

In [4]:
# Top 10 artists by playlist appearances

artist_performance.head(10)

,artist,playlist_appearances,unique_songs,average_position,average_popularity
242,Taylor Swift,2014,96,24.557100,87.096822
286,Zach Bryan,1854,35,25.619202,86.022114
177,Morgan Wallen,1666,10,30.610444,85.405762
229,Sabrina Carpenter,966,14,17.598344,90.198758
65,Drake,838,35,26.256563,84.608592
40,Chappell Roan,794,6,20.647355,88.664987
266,Travis Scott,760,31,25.744737,88.588158
22,Billie Eilish,729,12,20.362140,92.770919
273,"Tyler, The Creator",705,16,25.259574,88.394326
227,SZA,646,3,26.238390,91.668731


### Key Finding

The results show which artists had the strongest presence on the US Top 50 playlist based on their total playlist appearances.

## 3. Popularity Trends

Daily popularity measures help identify how the overall popularity of songs on the playlist changes over time.

In [ ]:
# Daily popularity trend

daily_popularity = (
    df.groupby("date")
    .agg(
        average_popularity=("popularity", "mean"),
        median_popularity=("popularity", "median")
    )
    .reset_index()
)

daily_popularity["popularity_trend_7d"] = (
    daily_popularity["average_popularity"]
    .rolling(7, min_periods=1)
    .mean()
)

daily_popularity.head(10)

,date,average_popularity,median_popularity,popularity_trend_7d
0,2024-05-18,90.06,90.0,90.060000
1,2024-05-19,90.00,90.0,90.030000
2,2024-05-20,90.04,90.0,90.033333
3,2024-05-21,88.68,90.0,89.695000
4,2024-05-22,88.16,90.0,89.388000
5,2024-05-23,88.58,90.0,89.253333
6,2024-05-24,89.02,90.0,89.220000
7,2024-05-25,89.82,90.0,89.185714
8,2024-05-26,89.96,90.0,89.180000
9,2024-05-27,89.84,90.0,89.151429


## 4. Release Format Analysis

This section compares songs from different release formats using chart position, popularity, and duration.

In [ ]:
# Performance by release format

release_format = (
    df.groupby("album_type")
    .agg(
        track_observations=("song", "count"),
        unique_songs=("song", "nunique"),
        average_position=("position", "mean"),
        average_popularity=("popularity", "mean"),
        average_duration_ms=("duration_ms", "mean")
    )
    .reset_index()
)

release_format["average_duration_minutes"] = (
    release_format["average_duration_ms"] / 60000
)

release_format

,album_type,track_observations,unique_songs,average_position,average_popularity,average_duration_ms,average_duration_minutes
0,album,19672,745,26.095821,86.667751,201714.586163,3.361910
1,compilation,42,8,33.190476,78.380952,200773.047619,3.346217
2,single,7986,239,23.991861,90.126597,191127.795768,3.185463


### Key Finding

The release-format comparison shows how album, single, and compilation tracks differ in their average chart position, popularity, and duration.

In [7]:
# Compare average popularity by release format

release_format[[
    "album_type",
    "average_position",
    "average_popularity",
    "average_duration_minutes"
]]

,album_type,average_position,average_popularity,average_duration_minutes
0,album,26.095821,86.667751,3.361910
1,compilation,33.190476,78.380952,3.346217
2,single,23.991861,90.126597,3.185463


## 5. Chart Position and Popularity

This section examines the relationship between playlist position and song popularity.

In [8]:
# Correlation between chart position and popularity

position_popularity_corr = df["position"].corr(
    df["popularity"],
    method="spearman"
)

print(
    f"Spearman correlation between chart position and popularity: "
    f"{position_popularity_corr:.3f}"
)

Spearman correlation between chart position and popularity: -0.245


## 6. Explicit vs Non-Explicit Tracks

This section compares explicit and non-explicit songs based on their average chart position and popularity.

In [ ]:
# Performance by explicit status

explicit_analysis = (
    df.groupby("is_explicit")
    .agg(
        track_observations=("song", "count"),
        unique_songs=("song", "nunique"),
        average_position=("position", "mean"),
        average_popularity=("popularity", "mean")
    )
    .reset_index()
)

explicit_analysis

,is_explicit,track_observations,unique_songs,average_position,average_popularity
0,False,14427,458,25.598531,87.981146
1,True,13273,499,25.392903,87.295035


## 7. Final Key Insights

The analysis highlights several important patterns in the US Top 50 playlist data:

- Some songs maintain chart presence for a long period, indicating sustained playlist performance.
- A small group of artists accounts for a large number of playlist appearances.
- Song popularity varies over time, while the 7-day trend helps show the overall direction.
- Album, single, and compilation tracks show differences in chart position, popularity, and duration.
- Chart position and popularity have a negative relationship, meaning songs appearing closer to the top generally have higher popularity values.
- Explicit and non-explicit tracks can be compared using their average chart position and popularity.

In [10]:
print("04_insights.ipynb completed successfully.")
print("Dataset shape:", df.shape)
print("Missing values:", df.isna().sum().sum())

04_insights.ipynb completed successfully.
Dataset shape: (27700, 10)
Missing values: 0
